# TD5 — Knowledge Reasoning over the RDF Graph

This session applies reasoning techniques to the Knowledge Base built in TD4.  

**Goals:**
- ✅ Load the expanded KB (`.ttl` from TD4)
- ✅ Run SPARQL queries to explore the graph
- ✅ Apply OWL/RDFS reasoning (with `owlrl`)
- ✅ Detect inverse, transitive, symmetric properties
- ✅ Perform path queries and entity similarity via shared predicates
- ✅ Export inferred triples

In [ ]:
# ── 0. Dependencies ────────────────────────────────────────────────────────
import importlib, subprocess, sys

def ensure(pkg, pip_name=None):
    pip_name = pip_name or pkg
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])

for p, n in [('rdflib', None), ('owlrl', None), ('pandas', None)]:
    ensure(p, n)

import re
from pathlib import Path
from collections import defaultdict

import pandas as pd
from rdflib import Graph, Namespace, OWL, RDF, RDFS, URIRef, Literal
import owlrl

print('Imports OK')

In [ ]:
# ── 1. Load the KB from TD4 ────────────────────────────────────────────────
def find_root(start: Path) -> Path:
    for p in [start.resolve()] + list(start.resolve().parents):
        if (p / 'td4').exists():
            return p
    raise FileNotFoundError('Cannot find workspace root')

ROOT = find_root(Path.cwd())
TD4  = ROOT / 'td4'
TD5  = ROOT / 'td5'
OUT  = TD5 / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

KB_TTL = TD4 / 'outputs' / 'expanded_kb.ttl'

g = Graph()
g.parse(str(KB_TTL), format='turtle')

MYKB = Namespace('http://private.org/energy/')
WDT  = Namespace('http://www.wikidata.org/prop/direct/')
WD   = Namespace('http://www.wikidata.org/entity/')
g.bind('mykb', MYKB)
g.bind('wdt', WDT)
g.bind('wd', WD)

print(f'Loaded: {KB_TTL}')
print(f'Triples in KB: {len(g):,}')

In [ ]:
# ── 2. Exploratory SPARQL queries ─────────────────────────────────────────
def sparql_df(graph: Graph, query: str, label: str = '') -> pd.DataFrame:
    """Run a SPARQL SELECT and return a DataFrame."""
    result = graph.query(query)
    cols   = [str(v) for v in result.vars]
    rows   = [tuple(str(cell) if cell else '' for cell in r) for r in result]
    df     = pd.DataFrame(rows, columns=cols)
    if label:
        print(f'--- {label} ({len(df)} rows) ---')
    return df

# Q1: Most frequent predicates
q1 = """
SELECT ?p (COUNT(*) AS ?cnt)
WHERE { ?s ?p ?o }
GROUP BY ?p
ORDER BY DESC(?cnt)
LIMIT 15
"""
df_pred = sparql_df(g, q1, 'Top predicates')
df_pred.head(15)

In [ ]:
# Q2: Entities with the most connections
q2 = """
SELECT ?s ?label (COUNT(?p) AS ?degree)
WHERE {
  ?s ?p ?o .
  OPTIONAL { ?s <http://www.w3.org/2000/01/rdf-schema#label> ?label . }
}
GROUP BY ?s ?label
ORDER BY DESC(?degree)
LIMIT 15
"""
df_hub = sparql_df(g, q2, 'Hub entities (highest degree)')
df_hub.head(15)

In [ ]:
# Q3: Example domain triple pattern — entities related to 'produce' / 'generate'
q3 = """
SELECT ?s ?slabel ?o ?olabel
WHERE {
  ?s <http://private.org/energy/prop/produce> ?o .
  OPTIONAL { ?s <http://www.w3.org/2000/01/rdf-schema#label> ?slabel }
  OPTIONAL { ?o <http://www.w3.org/2000/01/rdf-schema#label> ?olabel }
}
LIMIT 20
"""
df_q3 = sparql_df(g, q3, 'entity -[produce]-> entity')
df_q3.head(20)

In [ ]:
# Q4: owl:sameAs links (cross-graph alignment)
q4 = """
SELECT ?private ?external
WHERE {
  ?private <http://www.w3.org/2002/07/owl#sameAs> ?external .
}
LIMIT 20
"""
df_same = sparql_df(g, q4, 'owl:sameAs alignments')
df_same.head(20)

In [ ]:
# ── 3. OWL/RDFS Reasoning with owlrl ─────────────────────────────────────
# Add some OWL axioms so that the reasoner can infer new triples

# Declare produce and supply as equivalent (same meaning)
produce = MYKB['prop/produce']
supply  = MYKB['prop/supply']
g.add((produce, OWL.equivalentProperty, supply))

# Declare 'part_of' as owl:TransitiveProperty
part_of = MYKB['prop/include']
g.add((part_of, RDF.type, OWL.TransitiveProperty))

# Declare an InverseOf axiom
use_prop      = MYKB['prop/use']
used_in_prop  = MYKB['prop/used_in']
g.add((use_prop, OWL.inverseOf, used_in_prop))

triples_before = len(g)
print(f'Triples before reasoning: {triples_before:,}')

# Apply RDFS + partial OWL RL reasoning
owlrl.DeductiveClosure(owlrl.RDFS_Semantics, rdfs_closure=True, axiomatic_triples=False,
                        datatype_axioms=False).expand(g)

triples_after = len(g)
print(f'Triples after  reasoning: {triples_after:,}')
print(f'Newly inferred           : {triples_after - triples_before:,}')

In [ ]:
# Q5: Verify inverse property was inferred
q5 = """
SELECT ?o ?s
WHERE {
  ?o <http://private.org/energy/prop/used_in> ?s .
}
LIMIT 10
"""
df_inv = sparql_df(g, q5, 'Inferred inverseOf (used_in)')
df_inv.head(10)

In [ ]:
# ── 4. Entity Similarity via Shared Predicates ────────────────────────────
# Two entities are 'similar' if they share objects for the same predicate

q_sim = """
SELECT ?a ?b ?p (COUNT(?o) AS ?shared)
WHERE {
  ?a ?p ?o .
  ?b ?p ?o .
  FILTER(?a != ?b)
  FILTER(STRSTARTS(STR(?a), "http://private.org/energy/entity/"))
  FILTER(STRSTARTS(STR(?b), "http://private.org/energy/entity/"))
}
GROUP BY ?a ?b ?p
HAVING (?shared >= 2)
ORDER BY DESC(?shared)
LIMIT 20
"""
df_sim = sparql_df(g, q_sim, 'Similar entity pairs (shared predicate-objects ≥ 2)')
df_sim.head(20)

In [ ]:
# ── 5. Export enriched graph ───────────────────────────────────────────────
OUT_INFERRED_TTL = OUT / 'inferred_kb.ttl'
OUT_INFERRED_NT  = OUT / 'inferred_kb.nt'

import warnings; warnings.filterwarnings('ignore')
g.serialize(destination=str(OUT_INFERRED_TTL), format='turtle')
g.serialize(destination=str(OUT_INFERRED_NT),  format='nt', encoding='utf-8')

print(f'Saved enriched KB:')
print(f'  {OUT_INFERRED_TTL}')
print(f'  {OUT_INFERRED_NT}')
print(f'Total triples after reasoning: {len(g):,}')

---
## TD5 — Report Notes

### Reasoning performed
| Axiom | Property | Effect |
|---|---|---|
| `owl:TransitiveProperty` | `mykb:prop/include` | If A includes B and B includes C → A includes C |
| `owl:inverseOf` | `use` / `used_in` | If X uses Y → Y is used_in X |
| `owl:equivalentProperty` | `produce` / `supply` | Both map to same Wikidata P1056 |

### SPARQL results interpretation
- The hub analysis shows which energy organizations and regions are most frequently mentioned in triples.
- The similarity query identifies entity pairs that share the same predicates and objects — useful for deduplication or cluster analysis.

---
# Part 2: Knowledge Graph Embedding (KGE) with PyKEEN

In this section, we apply Knowledge Graph Embedding to the knowledge base constructed in TD4, as requested by the instructions.
We will:
1. Prepare the dataset (Train / Validation / Test splits)
2. Train at least two embedding models (e.g., TransE, ComplEx)
3. Evaluate using Link Prediction metrics (MRR, Hits@10)
4. Subsample the KB to test Size Sensitivity
5. Perform Embedding Analysis (Nearest Neighbors, 2D t-SNE Clustering)

In [ ]:
# ── 6. KGE Dependencies ──────────────────────────────────────────────────
for p, n in [('pykeen', None), ('torch', None), ('matplotlib', None), ('sklearn', 'scikit-learn')]:
    ensure(p, n)

import torch
import pykeen
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import scipy.spatial.distance

print('PyKEEN setup complete.')

In [ ]:
# ── 7. Data Preparation & Splitting ──────────────────────────────────────
# We use the generated N-Triples file as input for PyKEEN
if 'OUT_INFERRED_NT' not in locals():
    # Fallback to expanded_kb.nt from TD4 if OUT_INFERRED_NT isn't defined yet
    NT_FILE = TD4 / 'outputs' / 'expanded_kb.nt'
else:
    NT_FILE = OUT_INFERRED_NT if OUT_INFERRED_NT.exists() else (TD4 / 'outputs' / 'expanded_kb.nt')

# Load Triples
tf = TriplesFactory.from_path(str(NT_FILE))
print(f"Total Triples in Factory: {tf.num_triples}")
print(f"Entities: {tf.num_entities}")
print(f"Relations: {tf.num_relations}")

# Split into train: 80%, validation: 10%, test: 10%
training, validation, testing = tf.split([0.8, 0.1, 0.1], random_state=42)

print(f"Train split: {training.num_triples}")
print(f"Valid split: {validation.num_triples}")
print(f"Test  split: {testing.num_triples}")

In [ ]:
# ── 8. Train Embedding Models ────────────────────────────────────────────
# We will train two models: TransE (Translational) and ComplEx (Bilinear/Complex)
# (Using 5 epochs for quick execution; increase to 50+ for meaningful embeddings).

print("Training TransE...")
res_transe = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='TransE',
    model_kwargs=dict(embedding_dim=50),
    training_kwargs=dict(num_epochs=5, batch_size=256),
    random_seed=42,
    device='cpu' # Switch to 'gpu' if available
)

print("\nTraining ComplEx...")
res_complex = pipeline(
    training=training,
    validation=validation,
    testing=testing,
    model='ComplEx',
    model_kwargs=dict(embedding_dim=50),
    training_kwargs=dict(num_epochs=5, batch_size=256),
    random_seed=42,
    device='cpu'
)

print("Training completed.")

In [ ]:
# ── 9. Evaluation ────────────────────────────────────────────────────────
def print_eval(name, results):
    try:
        metric_results = results.metric_results.to_df()
        mrr = metric_results.loc[
            (metric_results['Side'] == 'both') & 
            (metric_results['Type'] == 'realistic') & 
            (metric_results['Metric'] == 'inverse_harmonic_mean_rank')
        ]['Value'].values
        
        hits10 = metric_results.loc[
            (metric_results['Side'] == 'both') & 
            (metric_results['Type'] == 'realistic') & 
            (metric_results['Metric'] == 'hits_at_10')
        ]['Value'].values
        
        print(f"--- {name} Results ---")
        print(f"  MRR     : {mrr[0] if len(mrr)>0 else 'N/A':.4f}")
        print(f"  Hits@10 : {hits10[0] if len(hits10)>0 else 'N/A':.4f}\n")
    except Exception as e:
        print(f"Could not process evaluation metrics for {name}: {e}")

print_eval("TransE", res_transe)
print_eval("ComplEx", res_complex)

In [ ]:
# ── 10. KB Size Sensitivity ──────────────────────────────────────────────
if training.num_triples > 20000:
    print("Sampling 20k triples for sensitivity analysis...")
    sub_triples = tf.triples[:20000]
    tf_sub = TriplesFactory.from_labeled_triples(sub_triples)
    train_sub, val_sub, test_sub = tf_sub.split([0.8, 0.1, 0.1], random_state=42)
    
    res_sub = pipeline(
        training=train_sub,
        validation=val_sub,
        testing=test_sub,
        model='TransE',
        model_kwargs=dict(embedding_dim=50),
        training_kwargs=dict(num_epochs=5, batch_size=256),
        random_seed=42,
        device='cpu'
    )
    print_eval("TransE on Subsample (20k)", res_sub)
else:
    print("KB is already <= 20k triples, skipping subsampling step.")

In [ ]:
# ── 11. Nearest Neighbors Analysis ────────────────────────────────────────
entity_embeddings = res_transe.model.entity_representations[0](indices=None).detach().cpu().numpy()
ent_to_id = tf.entity_to_id
id_to_ent = {v: k for k, v in ent_to_id.items()}

def get_nearest_neighbors(entity_name, k=5):
    if entity_name not in ent_to_id:
        print(f"Entity '{entity_name}' not found.")
        return
    idx = ent_to_id[entity_name]
    emb = entity_embeddings[idx].reshape(1, -1)
    
    distances = scipy.spatial.distance.cdist(emb, entity_embeddings, metric='cosine')[0]
    nearest_indices = np.argsort(distances)[:k+1] # +1 to skip self
    
    print(f"Nearest neighbors to '{entity_name}':")
    for i in nearest_indices:
        if i == idx: continue
        print(f" - {id_to_ent[i]} (dist: {distances[i]:.4f})")

sample_ent = id_to_ent.get(0, None)
if sample_ent:
    get_nearest_neighbors(sample_ent)

In [ ]:
# ── 12. Clustering Analysis (t-SNE) ───────────────────────────────────────
print("Running t-SNE... (this may take a moment)")
tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')

SAMPLE_SIZE = min(5000, len(entity_embeddings))
sample_indices = np.random.choice(len(entity_embeddings), SAMPLE_SIZE, replace=False)
sample_embeddings = entity_embeddings[sample_indices]

embeddings_2d = tsne.fit_transform(sample_embeddings)

colors = []
for idx in sample_indices:
    ent_uri = id_to_ent[idx]
    domain = ent_uri.split('/')[-1][:2] if '/' in ent_uri else 'unknown'
    colors.append(hash(domain) % 20)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=colors, cmap='tab20', s=10, alpha=0.7)
plt.title("2D t-SNE projection of TransE Entity Embeddings")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.colorbar(scatter, label='Class/Namespace proxy cluster')
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 2 Report: Critical Reflection & Reasoning

### Model Comparison
- **TransE** is highly effective at modeling 1-to-1 relations and composition via translation, but struggles with 1-to-N or symmetric relations due to continuous vector algebra enforcing contradictory states where relations collapse to null distances.
- **ComplEx** utilizes complex space vector representations, enabling natural mapping of symmetric and antisymmetric relations, performing logically coherent algebraic operations through dot products over complex fields.

### KB Size Sensitivity
- Due to the sparsity of training structure, small KBs generally produce unstable embeddings prone to overfitting on simple frequency heuristics.
- Model representations become statistically coherent against the true graph topography only as input depth (full dataset scale) expands, reinforcing accurate translatable distances across relations.

### Rule-based vs. Embedding-based Reasoning
- **Rule-based (SWRL / OWL):** Mechanizes deterministic reasoning, enforcing logical soundness. Axioms explicitly construct absolutely true edges on valid preconditions (Monotonic consistency).
- **Embedding-based (KGE):** Performs mathematical probabilistic extrapolations grounded on topology scoring. While lacking formal logical soundness, embeddings identify fuzzy relations predicting semantic continuity (`person + sibling + man ≈ brother`) regardless of whether formal schemas assert them in explicit paths, handling incomplete or open-world variations far more robustly.